# Processing Status Dashboard

Shows which tiles have been processed by reading:
- `tile_list.geojson` — the full set of land tiles (static, created by notebook 01)
- Icechunk commit history — the single source of truth for processed tiles

Set your Azure credentials before running:
```python
os.environ["AZURE_STORAGE_ACCOUNT"] = "..."
os.environ["AZURE_STORAGE_SAS_TOKEN"] = "..."
os.environ["AZURE_CONTAINER"] = "..."
os.environ["ICECHUNK_PREFIX"] = "modis-lst-demo"
```

In [ ]:
import os
import re
import sys
from pathlib import Path

import icechunk
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from utils import load_config, load_tile_list

cfg = load_config()

## Load tile list and check Icechunk history

In [ ]:
tile_gdf = load_tile_list()
land_set = {(int(r["row"]), int(r["col"])) for _, r in tile_gdf.iterrows()}

storage = icechunk.azure_storage(
    account=os.environ["AZURE_STORAGE_ACCOUNT"],
    container=os.environ["AZURE_CONTAINER"],
    prefix=os.environ["ICECHUNK_PREFIX"],
    sas_token=os.environ["AZURE_STORAGE_SAS_TOKEN"],
)
repo = icechunk.Repository.open(storage)

pattern = re.compile(r"tile_(\d+)_(\d+): processed")
processed = set()
for commit in repo.ancestry(branch="main"):
    m = pattern.match(commit.message)
    if m:
        processed.add((int(m.group(1)), int(m.group(2))))

remaining = land_set - processed

print(f"Land tiles:  {len(land_set)}")
print(f"Processed:   {len(processed)}")
print(f"Remaining:   {len(remaining)}")
print(f"Progress:    {100 * len(processed) / len(land_set):.1f}%")

## Visualize tile grid

In [ ]:
tile_gdf["status"] = tile_gdf.apply(
    lambda r: "processed" if (int(r["row"]), int(r["col"])) in processed else "unprocessed",
    axis=1,
)

color_map = {"processed": "#4caf50", "unprocessed": "#f5a623"}
fig, ax = plt.subplots(figsize=(14, 7))
for status, color in color_map.items():
    tile_gdf[tile_gdf["status"] == status].plot(ax=ax, color=color, edgecolor="white", linewidth=0.3, label=status)

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    f"Tile processing status — {len(processed)}/{len(land_set)} done "
    f"({100*len(processed)/len(land_set):.1f}%)"
)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## List unprocessed tiles

In [ ]:
if remaining:
    print("Unprocessed tiles (row, col):")
    for r, c in sorted(remaining):
        lat_min = -90 + r * cfg["TILE_SIZE_DEG"]
        lon_min = -180 + c * cfg["TILE_SIZE_DEG"]
        print(f"  row={r}, col={c}  lat=[{lat_min:.0f}, {lat_min+10:.0f}]  lon=[{lon_min:.0f}, {lon_min+10:.0f}]")
else:
    print("All land tiles processed.")